##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input


(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

vgg_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input, name="vgg16_preprocess"),
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, name="predictions")                          
], name="cifar10_vgg16")

print("Model Architecture ")
vgg_model.summary()

print("\nTrainable layers in backbone:", sum(l.trainable for l in vgg_base.layers), "/", len(vgg_base.layers))

vgg_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print("\nTraining Feature Extractor")
history = vgg_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss, test_acc = vgg_model.evaluate(x_test, y_test, verbose=0)
print("\nVGG16 (frozen) test accuracy:", test_acc)
print("VGG16 (frozen) test loss:", test_loss)

vgg_base.trainable = True

for layer in vgg_base.layers[:-4]:
    layer.trainable = False

print("\nTrainable layers in backbone after unfreezing:", sum(l.trainable for l in vgg_base.layers), "/", len(vgg_base.layers))

vgg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print("\nFine-Tuning ")
history_ft = vgg_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = vgg_model.evaluate(x_test, y_test, verbose=0)
print("\nVGG16 (fine-tuned) test accuracy:", test_acc_ft)
print("VGG16 (fine-tuned) test loss:", test_loss_ft)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model Architecture 


Model: "cifar10_vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16_preprocess (Lambda)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,719,818 (56.15 MB)

 Trainable params: 5,130 (20.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)


Trainable layers in backbone: 0 / 19

Training Feature Extractor
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 247s 333ms/step - accuracy: 0.4798 - loss: 1.7101 - val_accuracy: 0.7908 - val_loss: 0.6471
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 242s 343ms/step - accuracy: 0.7114 - loss: 0.8356 - val_accuracy: 0.8170 - val_loss: 0.5626
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 242s 344ms/step - accuracy: 0.7373 - loss: 0.7663 - val_accuracy: 0.8256 - val_loss: 0.5281

VGG16 (frozen) test accuracy: 0.8227999806404114
VGG16 (frozen) test loss: 0.5444982647895813

Trainable layers in backbone after unfreezing: 4 / 19

Fine-Tuning 
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 285s 400ms/step - accuracy: 0.7635 - loss: 0.6919 - val_accuracy: 0.8660 - val_loss: 0.3997
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 280s 398ms/step - accuracy: 0.8210 - loss: 0.5312 - val_accuracy: 0.8842 - val_loss: 0.3583
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 281s 399ms/step - accuracy: 0.8408 - loss: 0.4506 - val_accuracy: 0.8890 - v